# Life-Long Learning: Catastrophic Forgetting, the Multi-task Upper Bound, EWC, and How to Measure Forgetting

**The central problem (Hung-yi Lee, Life-Long Learning).** When a neural network is trained on a *new* task, it tends to forget the *old* ones — even though the very same network, trained on all tasks together, can solve them all. As the lecture puts it: *"it is not that the machine is unable, it just did not do it."* The failure is one of **learning strategy**, not of capacity.

**The storyline of this notebook.** We follow a single arc, end-to-end, on a free Colab CPU:

1. **Problem — Catastrophic forgetting (LO1, C08–C10).** One small MLP trained sequentially on several Permuted-MNIST tasks loses accuracy on earlier tasks.
2. **Upper bound — Multi-task training (LO2, C11–C13).** Train on all tasks jointly to prove the capacity is sufficient; this is the practical ceiling.
3. **Solution — EWC / Selective Synaptic Plasticity (LO3–LO4, C14–C17).** Guard important weights with the elastic penalty $L'(\theta)=L(\theta)+\lambda\sum_i b_i(\theta_i-\theta_{i,\text{old}})^2$, and interactively feel the forgetting-vs-intransigence trade-off.
4. **Measurement — The R matrix (LO5, C18–C20).** Turn every claim into a number: Accuracy, Backward Transfer (forgetting, usually negative), and Forward Transfer.

**Demo philosophy.** One small MLP and one dataset family (Permuted-MNIST) run every experiment, top to bottom, in a single class session.

In [ ]:
# C02 — Colab-ready environment + global config (the single setup cell).
# ipywidgets is the only dependency we may need to install; torch/torchvision/numpy/matplotlib
# are preinstalled on Colab.
import os, sys, random, subprocess

try:
    import ipywidgets  # noqa: F401
except ImportError:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'], check=True)
    except Exception as e:
        print(f'[warn] could not install ipywidgets ({e}); interactive cells will use static fallbacks.')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
try:
    import ipywidgets as widgets
except Exception:
    widgets = None

# Enable inline plotting inside IPython/Colab; silently skipped in plain Python / CI.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed: int = 0) -> None:
    '''Seed python, numpy and torch (incl. cuda) for reproducible runs.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(0)

# Global hyperparameters reused everywhere (small enough for a classroom CPU session).
NUM_TASKS = 3
HIDDEN = 256
EPOCHS_PER_TASK = 2
BATCH_SIZE = 128
SUBSET_SIZE = 6000
TEST_SUBSET_SIZE = 2000
LR = 1e-3

# Quick self-checks.
assert isinstance(NUM_TASKS, int) and NUM_TASKS >= 2
assert HIDDEN > 0 and EPOCHS_PER_TASK >= 1 and BATCH_SIZE > 0 and SUBSET_SIZE > 0
assert str(device) in ('cpu', 'cuda')
set_seed(0); _a = torch.randn(5); set_seed(0); _b = torch.randn(5)
assert torch.allclose(_a, _b)

print(f'torch {torch.__version__} | torchvision {torchvision.__version__} | device={device}')

## Experimental setup: Permuted-MNIST

We use **Permuted-MNIST** as a clean stand-in for the lecture's sequence of tasks. Each task applies its **own fixed random permutation of the 784 pixels** to every MNIST image:

- All tasks share the **same label space** (digits 0–9) and the **same difficulty**.
- But each task requires a **different input-to-feature mapping**, because the pixels are shuffled differently.

This makes forgetting both **easy to induce** and **easy to see**: a model tuned for task *k*'s pixel layout is mis-aligned for task *k+1*. We keep **task 0 as the identity permutation** (an intuitive anchor) and reuse one small MLP throughout, matching the lecture's small-network setup and holding the model class and dataset fixed for comparability and speed. *(Maps to LO1.)*

In [ ]:
# C04 — Build the Permuted-MNIST tasks once (reused by every training and viewing cell).
from torch.utils.data import DataLoader, TensorDataset

transform = transforms.Compose([transforms.ToTensor()])  # images -> float in [0,1]

def _load_mnist():
    train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    return train, test

try:
    mnist_train, mnist_test = _load_mnist()
except Exception as e:
    print(f'[warn] MNIST download failed ({e}); retrying once...')
    mnist_train, mnist_test = _load_mnist()

# Materialize flattened tensors in [0,1], shape (N, 784).
X_train_full = mnist_train.data.float().div(255.0).view(-1, 784)
y_train_full = mnist_train.targets.clone()
X_test_full = mnist_test.data.float().div(255.0).view(-1, 784)
y_test_full = mnist_test.targets.clone()

# Subsample for speed with fixed indices (reproducible across reruns).
set_seed(0)
n_tr = min(SUBSET_SIZE, X_train_full.shape[0])
n_te = min(TEST_SUBSET_SIZE, X_test_full.shape[0])
tr_idx = torch.randperm(X_train_full.shape[0])[:n_tr]
te_idx = torch.randperm(X_test_full.shape[0])[:n_te]
X_train_base = X_train_full[tr_idx].contiguous()
y_train_sub = y_train_full[tr_idx].contiguous()
X_test_base = X_test_full[te_idx].contiguous()
y_test_sub = y_test_full[te_idx].contiguous()

# Fixed pixel permutations: task 0 = identity anchor; tasks >= 1 are random bijections.
permutations = [torch.arange(784)]
for k in range(1, NUM_TASKS):
    g = torch.Generator().manual_seed(1000 + k)
    permutations.append(torch.randperm(784, generator=g))

assert torch.equal(permutations[0], torch.arange(784))
for k in range(1, NUM_TASKS):
    assert sorted(permutations[k].tolist()) == list(range(784))

def permute_pixels(x: torch.Tensor, perm: torch.Tensor) -> torch.Tensor:
    '''Apply a fixed pixel permutation to flattened images (..., 784).'''
    return x[..., perm]

# Build the tasks list: each task carries its own permuted train/test tensors.
tasks = []
for k in range(NUM_TASKS):
    tasks.append({
        'train_x': permute_pixels(X_train_base, permutations[k]),
        'train_y': y_train_sub,
        'test_x': permute_pixels(X_test_base, permutations[k]),
        'test_y': y_test_sub,
        'perm': permutations[k],
        'perm_index': k,
    })

# Raw (identity) flattened test images for the permutation viewer.
x_test_raw = X_test_base

def get_task_loader(task: dict, batch_size: int = BATCH_SIZE, train: bool = True, shuffle: bool = True):
    x = task['train_x'] if train else task['test_x']
    y = task['train_y'] if train else task['test_y']
    return DataLoader(TensorDataset(x, y), batch_size=batch_size, shuffle=shuffle)

# Self-checks.
assert len(tasks) == NUM_TASKS and len(permutations) == NUM_TASKS
for t in tasks:
    assert t['train_x'].shape[1] == 784
    assert float(t['train_x'].min()) >= 0.0 and float(t['train_x'].max()) <= 1.0
assert int(tasks[0]['train_y'].max()) <= 9 and int(tasks[0]['train_y'].min()) >= 0
assert torch.equal(tasks[0]['train_x'], X_train_base)  # task 0 is unpermuted
assert x_test_raw.shape[-1] == 784
print(f'Built {NUM_TASKS} Permuted-MNIST tasks | train subset={n_tr}, test subset={n_te} per task')

In [ ]:
# C05 — Interactive permutation viewer: see why each task needs a different mapping.
def show_permutation(task_id: int, sample_id: int) -> None:
    task_id = int(max(0, min(task_id, NUM_TASKS - 1)))
    if X_test_base.shape[0] == 0:
        print('[warn] empty test subset; nothing to show.')
        return
    sample_id = int(max(0, min(sample_id, X_test_base.shape[0] - 1)))
    orig = X_test_base[sample_id].reshape(28, 28).numpy()
    perm_img = X_test_base[sample_id][permutations[task_id]].reshape(28, 28).numpy()
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(orig, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(perm_img, cmap='gray')
    axes[1].set_title(f'Task {task_id} permutation'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

# Build the slider widget when ipywidgets is available.
if widgets is not None:
    try:
        widgets.interact(
            show_permutation,
            task_id=widgets.IntSlider(min=0, max=NUM_TASKS - 1, value=1, description='task_id'),
            sample_id=widgets.IntSlider(min=0, max=X_test_base.shape[0] - 1, value=7, description='sample_id'),
        )
    except Exception as e:
        print(f'[warn] widget init failed ({e}); showing static default.')

# Always render a sensible static default so the cell produces output under Run All.
show_permutation(1, 7)

## The shared model and training machinery

Every experiment reuses **one small MLP**: $784 \to \text{HIDDEN} \to \text{HIDDEN} \to 10$ (two hidden layers, in the spirit of the lecture's 3-layer net). Holding the architecture fixed means **capacity is constant** — only the *learning strategy* changes between experiments, so any difference in retention is the strategy's doing, not the model's size.

We define three shared helpers once:

- `train_on_task` — train the model on a single task (optionally with an extra penalty term, used later by EWC);
- `evaluate` — test accuracy on a single task;
- `build_R_matrix` — train a fresh model sequentially across all tasks and record accuracy on **every** task after **every** stage.

The R matrix includes a **random-init baseline row (row 0)** so that **Forward Transfer** is computable later. *(Maps to LO1.)*

In [ ]:
# C07 — Shared model + reusable train / evaluate / R-matrix utilities (defined once).
class MLP(nn.Module):
    def __init__(self, in_dim: int = 784, hidden: int = HIDDEN, out_dim: int = 10) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() > 2:
            x = x.view(x.size(0), -1)
        return self.net(x)  # logits, no softmax

def train_on_task(model, task, epochs=EPOCHS_PER_TASK, lr=LR, extra_loss_fn=None):
    '''Train `model` in place on one task with Adam + cross-entropy.
    extra_loss_fn(model) -> scalar tensor is ADDED to the loss (used by EWC).'''
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = get_task_loader(task, BATCH_SIZE, train=True, shuffle=True)
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(model(x), y)
            if extra_loss_fn is not None:
                loss = loss + extra_loss_fn(model)
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate(model, task):
    '''Return test accuracy in [0,1] on one task (no_grad, eval mode).'''
    model.eval()
    loader = get_task_loader(task, BATCH_SIZE, train=False, shuffle=False)
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += int((pred == y).sum().item())
            total += int(y.size(0))
    model.train()
    return float(correct / total) if total > 0 else 0.0

def build_R_matrix(train_step_fn, tasks, num_tasks=NUM_TASKS):
    '''Train a fresh model sequentially; record accuracy on every task after each stage.
    Returns R of shape (num_tasks+1, num_tasks); row 0 = random-init baseline.
    train_step_fn(model, task, task_index) must train the model in place.'''
    set_seed(0)
    model = MLP().to(device)
    R = np.zeros((num_tasks + 1, num_tasks))
    for j in range(num_tasks):
        R[0, j] = evaluate(model, tasks[j])
    for i, task in enumerate(tasks):
        train_step_fn(model, task, i)
        for j in range(num_tasks):
            R[i + 1, j] = evaluate(model, tasks[j])
    return R

# Self-checks.
_m = MLP().to(device)
assert _m(torch.randn(4, 784).to(device)).shape == (4, 10)
assert 0.0 <= evaluate(MLP().to(device), tasks[0]) <= 1.0
_R = build_R_matrix(lambda model, task, i: train_on_task(model, task, epochs=1), tasks)
assert _R.shape == (NUM_TASKS + 1, NUM_TASKS) and np.all(np.isfinite(_R)) and np.all((_R >= 0) & (_R <= 1))
assert float(np.mean(_R[0])) < 0.5  # baseline row near chance before training
print('Model + utilities ready. Smoke-test R matrix shape:', _R.shape)

## Experiment 1 — Naive sequential training and catastrophic forgetting

**Predict before you run.** We train one model on task 0, then task 1, then task 2, evaluating on *all* tasks after each stage. Expect:

- The **diagonal** of the R matrix stays **high** — the model *can* still learn each new task (capacity is fine).
- Accuracy on **earlier** tasks **collapses** as later tasks are learned — the model overwrites the weights those tasks relied on.

This is the lecture's core demonstration: the network **could** retain every task (we prove it with multi-task training next), it simply **did not**. The failure is one of strategy, not capacity. *(Maps to LO1.)*

In [ ]:
# C09 — Catastrophic forgetting: naive sequential training fills R_sgd.
def sgd_step(model, task, task_index):
    '''Plain training step: no regularization, just train on the current task.'''
    return train_on_task(model, task, epochs=EPOCHS_PER_TASK)

try:
    R_sgd = build_R_matrix(sgd_step, tasks)
except Exception as e:
    print(f'[error] naive sequential training failed: {e}')
    raise

print('R_sgd  (rows = accuracy after training r tasks; row 0 = random-init baseline; cols = task j)')
print(np.round(R_sgd, 3))

assert R_sgd.shape == (NUM_TASKS + 1, NUM_TASKS)
assert np.all((R_sgd >= 0) & (R_sgd <= 1))

In [ ]:
# C10 — Forgetting curve: each task's accuracy vs how many tasks have been learned.
assert R_sgd.shape == (NUM_TASKS + 1, NUM_TASKS), 'Run C09 first to build R_sgd.'

stages = np.arange(1, NUM_TASKS + 1)  # training-stage index (skip baseline row 0)
plt.figure(figsize=(6, 4))
for j in range(NUM_TASKS):
    plt.plot(stages, R_sgd[1:, j], marker='o', label=f'Task {j}')
plt.xlabel('# tasks learned so far')
plt.ylabel('test accuracy')
plt.xticks(stages)
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.title('Catastrophic forgetting: early-task accuracy collapses')
plt.tight_layout(); plt.show()

## Experiment 2 — Multi-task (joint) training: the upper bound

If we train on **all tasks' data at once** (shuffled together), there is no "later task" overwriting an "earlier" one, so forgetting essentially vanishes. Multi-task training is therefore the **practical upper bound** against which every continual-learning method is measured — it proves the network's capacity is sufficient for all tasks simultaneously.

The catch, and the reason dedicated life-long-learning methods exist, is that joint training requires **storing all data forever** and revisiting it — exactly the storage/compute cost the lecture flags. A true continual learner must approach this ceiling **without** keeping all old data. *(Maps to LO2.)*

In [ ]:
# C12 — Multi-task (joint) training: the practical upper bound.
from torch.utils.data import DataLoader, TensorDataset

def build_joint_loader(tasks, batch_size=BATCH_SIZE):
    '''Concatenate all tasks' train data into one shuffled loader.'''
    X = torch.cat([t['train_x'] for t in tasks], dim=0)
    Y = torch.cat([t['train_y'] for t in tasks], dim=0)
    return DataLoader(TensorDataset(X, Y), batch_size=batch_size, shuffle=True)

set_seed(0)
mt_model = MLP().to(device)
opt = torch.optim.Adam(mt_model.parameters(), lr=LR)
joint_loader = build_joint_loader(tasks, BATCH_SIZE)

# The combined set is NUM_TASKS x larger, so EPOCHS_PER_TASK epochs is a fair-or-stronger upper bound.
mt_model.train()
for _ in range(EPOCHS_PER_TASK):
    for x, y in joint_loader:
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(mt_model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()

multitask_acc = np.array([evaluate(mt_model, t) for t in tasks])
print('Multi-task per-task accuracy:', np.round(multitask_acc, 3))
print('Multi-task mean accuracy   :', round(float(multitask_acc.mean()), 3))

assert multitask_acc.shape == (NUM_TASKS,)
assert np.all((multitask_acc >= 0) & (multitask_acc <= 1))

In [ ]:
# C13 — Same capacity, different retention: naive final vs multi-task upper bound.
assert len(R_sgd[NUM_TASKS, :]) == len(multitask_acc) == NUM_TASKS

naive_final = R_sgd[NUM_TASKS, :]
upper = multitask_acc
x = np.arange(NUM_TASKS)
width = 0.38

plt.figure(figsize=(6, 4))
plt.bar(x - width / 2, naive_final, width, label='Naive sequential (final)')
plt.bar(x + width / 2, upper, width, label='Multi-task upper bound')
plt.xticks(x, [f'Task {j}' for j in range(NUM_TASKS)])
plt.ylim(0, 1)
plt.ylabel('test accuracy')
plt.title('Same capacity, different retention')
plt.legend()
plt.tight_layout(); plt.show()

## Experiment 3 — Selective Synaptic Plasticity via EWC

The lecture's most detailed solution protects the weights that matter to old tasks. **Elastic Weight Consolidation (EWC)** adds a quadratic guard to the loss:

$$L'(\theta) = L(\theta) + \lambda \sum_i b_i\,(\theta_i - \theta_{i,\text{old}})^2$$

- $L(\theta)$ — the ordinary loss on the **new** task.
- $\theta_{i,\text{old}}$ — the value of parameter $i$ **after** the previous tasks were consolidated.
- $b_i$ — the **guard** / importance of parameter $i$, estimated by the **diagonal Fisher information**. Large $b_i$ ⇒ the weight mattered to old tasks ⇒ penalize moving it.
- $\lambda$ — the global strength of the guard.

**Two extremes:**

- $b_i = 0$ (or $\lambda = 0$): no protection ⇒ full **catastrophic forgetting**.
- $b_i \to \infty$ (or huge $\lambda$): weights are frozen ⇒ **intransigence** — the model cannot learn the new task.

EWC is the canonical representative of the **Selective Synaptic Plasticity** family (others: SI, MAS, RWalk). *(Maps to LO3.)*

In [ ]:
# C15 — EWC primitives: Fisher importance, parameter snapshot, quadratic guard, EWC training.
def compute_fisher(model, task, num_samples=200):
    '''Diagonal (empirical) Fisher importance b_i from per-sample squared gradients.'''
    model.eval()
    fisher = {n: torch.zeros_like(p) for n, p in model.named_parameters() if p.requires_grad}
    n_total = task['train_x'].shape[0]
    count = min(num_samples, n_total)
    for idx in range(count):
        x = task['train_x'][idx:idx + 1].to(device)
        y = task['train_y'][idx:idx + 1].to(device)
        model.zero_grad()
        logp = F.log_softmax(model(x), dim=1)
        loss = F.nll_loss(logp, y)  # empirical Fisher under the true label
        loss.backward()
        for n, p in model.named_parameters():
            if p.requires_grad and p.grad is not None:
                fisher[n] += p.grad.detach() ** 2
    for n in fisher:
        fisher[n] /= max(count, 1)
    model.zero_grad()
    model.train()
    return {n: v.detach() for n, v in fisher.items()}

def snapshot_params(model):
    '''Store consolidated theta_i_old after a task.'''
    return {n: p.clone().detach() for n, p in model.named_parameters() if p.requires_grad}

def ewc_penalty(model, fisher_list, params_list):
    '''Quadratic guard summed over all previously consolidated tasks.'''
    if not fisher_list:
        return torch.zeros((), device=device)
    total = torch.zeros((), device=device)
    for fisher, params in zip(fisher_list, params_list):
        for n, p in model.named_parameters():
            if n in fisher:
                total = total + (fisher[n] * (p - params[n]) ** 2).sum()
    return total

def train_on_task_ewc(model, task, lam, fisher_list, params_list, epochs=EPOCHS_PER_TASK, lr=LR):
    '''Standard training with lam * EWC penalty added to the task loss.'''
    return train_on_task(
        model, task, epochs=epochs, lr=lr,
        extra_loss_fn=lambda m: lam * ewc_penalty(m, fisher_list, params_list),
    )

# Self-checks.
_m = MLP().to(device)
_f = compute_fisher(_m, tasks[0], num_samples=32)
assert set(_f.keys()) == set(n for n, _ in _m.named_parameters())
assert all(float(v.min()) >= 0.0 for v in _f.values())
_p0 = snapshot_params(_m)
assert 0.0 <= float(ewc_penalty(_m, [_f], [_p0])) < 1e-4  # same params => ~0 penalty
with torch.no_grad():
    list(_m.parameters())[0].add_(1.0)
assert float(ewc_penalty(_m, [_f], [_p0])) > 0.0  # moving a weight grows the penalty
assert float(ewc_penalty(_m, [], [])) == 0.0  # no consolidated tasks => zero penalty
print('EWC primitives ready (Fisher, snapshot, penalty, EWC training).')

In [ ]:
# C16 — Run EWC and precompute the lambda sweep so C17 never retrains live.
def make_ewc_step(lam):
    '''Stateful sequential EWC step with its own consolidation history.
    Consolidation (snapshot + Fisher) happens strictly AFTER each task's training.'''
    fisher_list, params_list = [], []
    def step(model, task, task_index):
        train_on_task_ewc(model, task, lam, fisher_list, params_list)
        params_list.append(snapshot_params(model))
        fisher_list.append(compute_fisher(model, task))
        return model
    return step

def ewc_run(lam):
    '''Build a full R matrix for sequential EWC at strength lam (build_R_matrix reseeds).'''
    return build_R_matrix(make_ewc_step(lam), tasks)

lambda_grid = [0.0, 1.0, 100.0, 10000.0]  # 0 = forgetting ... 10000 = intransigence
DEFAULT_LAMBDA = 100.0
assert DEFAULT_LAMBDA in lambda_grid

lambda_results = {}
for lam in lambda_grid:
    try:
        lambda_results[lam] = ewc_run(lam)
    except Exception as e:
        print(f'[error] EWC run failed at lambda={lam}: {e}')
        raise

R_ewc = lambda_results[DEFAULT_LAMBDA]
print(f'R_ewc at lambda={DEFAULT_LAMBDA}  (rows = after training r tasks; row 0 = baseline; cols = task j)')
print(np.round(R_ewc, 3))

assert set(lambda_results.keys()) == set(lambda_grid)
for lam in lambda_grid:
    R = lambda_results[lam]
    assert R.shape == (NUM_TASKS + 1, NUM_TASKS) and np.all((R >= 0) & (R <= 1))
assert R_ewc is lambda_results[DEFAULT_LAMBDA]

In [ ]:
# C17 — Star widget: lambda slider over PRECOMPUTED results (forgetting vs intransigence).
def show_lambda(lam):
    # Snap to the nearest grid key so a missing value never KeyErrors.
    if lam not in lambda_results:
        lam = min(lambda_grid, key=lambda g: abs(g - lam))
    R = lambda_results[lam]
    final = R[NUM_TASKS, :]
    old_acc = float(np.mean(final[:-1])) if NUM_TASKS >= 2 else float('nan')  # retention
    new_acc = float(final[-1])  # plasticity on the newest task
    avg = float(np.mean(final))
    labels = ['old-task retention', 'new-task accuracy', 'overall avg']
    values = [old_acc, new_acc, avg]
    plt.figure(figsize=(6, 4))
    bars = plt.bar(labels, values, color=['#4c72b0', '#dd8452', '#55a868'])
    for b, v in zip(bars, values):
        plt.text(b.get_x() + b.get_width() / 2, min(v + 0.02, 0.98), f'{v:.2f}', ha='center', va='bottom')
    plt.ylim(0, 1)
    plt.ylabel('test accuracy')
    plt.title(f'EWC trade-off at lambda = {lam}')
    plt.tight_layout(); plt.show()

if widgets is not None:
    try:
        widgets.interact(
            show_lambda,
            lam=widgets.SelectionSlider(options=list(lambda_grid), value=DEFAULT_LAMBDA, description='lambda'),
        )
    except Exception as e:
        print(f'[warn] widget init failed ({e}); showing static default.')

# Always render a static default so the cell produces output under Run All.
show_lambda(DEFAULT_LAMBDA)

## Evaluation framework: the R matrix and three metrics

Let $R_{i,j}$ be the test accuracy on **task $j$** after training on the first **$i$** tasks (row 0 = random-init baseline, before any training). For $T$ tasks the lecture defines:

- **Accuracy** $= \dfrac{1}{T}\sum_{i=1}^{T} R_{T,i}$ — mean of the **last row** (final accuracy across all tasks).
- **Backward Transfer (BWT)** $= \dfrac{1}{T-1}\sum_{i=1}^{T-1}\left(R_{T,i} - R_{i,i}\right)$ — how learning later tasks changed earlier ones. **Negative ⇒ forgetting.**
- **Forward Transfer (FWT)** $= \dfrac{1}{T-1}\sum_{i=2}^{T}\left(R_{i-1,i} - R_{0,i}\right)$ — how already-learned tasks help a *new* task, measured against the **random-init baseline row 0**.

These turn every qualitative claim above into a number. Expect **BWT strongly negative for naive SGD** and **closer to zero for EWC**. *(Maps to LO5.)*

In [ ]:
# C19 — Metrics: Accuracy, Backward Transfer, Forward Transfer + comparison table.
# R convention: shape (T+1, T); row 0 = baseline (0 tasks trained); row r in 1..T = after r tasks.
def accuracy(R):
    '''Mean accuracy over all tasks after the final training stage (mean of last row).'''
    return float(np.mean(R[-1, :]))

def backward_transfer(R):
    '''Mean (final acc - acc right after learning each earlier task). Negative => forgetting.'''
    T = R.shape[1]
    if T < 2:
        raise ValueError('Backward Transfer needs at least 2 tasks.')
    return float(np.mean([R[T, i] - R[i + 1, i] for i in range(T - 1)]))

def forward_transfer(R):
    '''Mean (acc on task i just before training it - random-init baseline row 0).'''
    T = R.shape[1]
    if T < 2:
        raise ValueError('Forward Transfer needs at least 2 tasks.')
    return float(np.mean([R[i, i] - R[0, i] for i in range(1, T)]))

def build_metrics_table(R_sgd, R_ewc, multitask_acc):
    rows = [
        {'Method': 'Naive SGD', 'Accuracy': round(accuracy(R_sgd), 3),
         'BWT': round(backward_transfer(R_sgd), 3), 'FWT': round(forward_transfer(R_sgd), 3)},
        {'Method': 'Multi-task (upper bound)', 'Accuracy': round(float(multitask_acc.mean()), 3),
         'BWT': 'N/A', 'FWT': 'N/A'},
        {'Method': 'EWC', 'Accuracy': round(accuracy(R_ewc), 3),
         'BWT': round(backward_transfer(R_ewc), 3), 'FWT': round(forward_transfer(R_ewc), 3)},
    ]
    try:
        import pandas as pd
        return pd.DataFrame(rows)[['Method', 'Accuracy', 'BWT', 'FWT']]
    except ImportError:
        return rows

metrics_table = build_metrics_table(R_sgd, R_ewc, multitask_acc)

if isinstance(metrics_table, list):  # pandas unavailable -> formatted text table
    header = f"{'Method':<26}{'Accuracy':>10}{'BWT':>8}{'FWT':>8}"
    print(header); print('-' * len(header))
    for r in metrics_table:
        print(f"{r['Method']:<26}{str(r['Accuracy']):>10}{str(r['BWT']):>8}{str(r['FWT']):>8}")
else:
    try:
        from IPython.display import display
        display(metrics_table)
    except Exception:
        print(metrics_table.to_string(index=False))

print('\nNote: BWT < 0 means forgetting; EWC should land closer to zero than Naive SGD.')

# Self-checks.
assert abs(accuracy(R_sgd) - float(np.mean(R_sgd[NUM_TASKS, :]))) < 1e-9
assert 0.0 <= accuracy(R_sgd) <= 1.0 and 0.0 <= accuracy(R_ewc) <= 1.0
assert (len(metrics_table) == 3) if isinstance(metrics_table, list) else (metrics_table.shape[0] == 3)

In [ ]:
# C20 — Side-by-side R-matrix heatmaps: naive SGD vs EWC (shared colorbar).
assert R_sgd.shape == R_ewc.shape == (NUM_TASKS + 1, NUM_TASKS)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
im = None
for ax, (R, name) in zip(axes, [(R_sgd, 'Naive SGD'), (R_ewc, 'EWC')]):
    im = ax.imshow(R, vmin=0, vmax=1, cmap='viridis', aspect='auto')
    ax.set_xticks(range(NUM_TASKS))
    ax.set_xticklabels([f'T{j}' for j in range(NUM_TASKS)])
    ax.set_yticks(range(NUM_TASKS + 1))
    ax.set_yticklabels(['init'] + [f'after T{r}' for r in range(NUM_TASKS)])
    ax.set_title(name)
    ax.set_xlabel('evaluated task')
    for r in range(NUM_TASKS + 1):
        for c in range(NUM_TASKS):
            val = R[r, c]
            ax.text(c, r, f'{val:.2f}', ha='center', va='center',
                    color='white' if val < 0.5 else 'black', fontsize=8)
fig.colorbar(im, ax=list(axes), fraction=0.046, pad=0.04)
fig.suptitle('R-matrix: rows = after training r tasks, cols = evaluated task')
plt.show()

## Recap and where to go next

**The arc we walked:**

- **Catastrophic forgetting (C08–C10).** A single MLP trained sequentially on Permuted-MNIST kept its diagonal high but lost early-task accuracy — it *could*, but *did not*.
- **Multi-task upper bound (C11–C13).** Joint training recovered all tasks, proving capacity was never the problem — but it needs all data stored forever.
- **EWC / selective synaptic plasticity (C14–C17).** The guarded loss $L'=L+\lambda\sum_i b_i(\theta_i-\theta_{i,\text{old}})^2$ closed much of the gap to the upper bound **without storing old data**, and the $\lambda$ slider let us feel the forgetting-vs-intransigence trade-off.
- **Quantitative evaluation (C18–C20).** The R matrix made it numeric: EWC's Backward Transfer is closer to zero than naive SGD's, and its heatmap keeps a brighter lower-left region.

**Key takeaway per objective:** capacity is sufficient (LO1–LO2); selectively guarding important weights via Fisher-weighted penalties recovers most of the lost retention (LO3–LO4); and the R-matrix metrics make the improvement measurable (LO5).

**Intentionally left out to keep this fast and self-contained, and where they fit:**

- **Memory / generative replay** — store or synthesize old-task data instead of regularizing.
- **Gradient Episodic Memory (GEM)** — constrain gradients so they never increase old-task loss.
- **Additional-resource methods** — Progressive Networks, PackNet, CPG grow or partition capacity per task.
- **Life-long learning vs. transfer learning** — LLL cares about *not forgetting*; transfer cares only about the new task.
- **Task ordering / curriculum** — the order in which tasks arrive can change the outcome.

Try raising `NUM_TASKS`, `EPOCHS_PER_TASK`, or `SUBSET_SIZE`, or implementing one of the methods above. *(Maps to all LOs.)*